
# ED Pathway Orchestrator — v12 (adds **provenance & artifact logging**)

Goal: **avoid loss of information**. v12 keeps all v11 logic (ICU constraints, approvals, agents, boarding)
and adds:
- A **RunLogger** that captures policy, capacities, approvals, timeline, watcher alerts, orders, and summary.
- A **Run Manifest** with seeds, versions, and hashes.
- **Artifact export** to `/mnt/data/ed_run_<ts>/...` and a one-click ZIP you can download.

> Tip: run this notebook top-to-bottom once. Artifacts will be saved even if you reset later.


In [ ]:

import time, math, json, random, yaml, hashlib, os, zipfile
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Literal, Any, Set, Iterable, Tuple, Callable

import numpy as np, pandas as pd
import torch, torch.nn as nn

SEED = 2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def safe_clip(obj, lower=None, upper=None): return obj.clip(lower=lower, upper=upper)
def safe_assign(df: pd.DataFrame, col: str, values): df = df.copy(); df.loc[:, col] = values; return df
def assert_no_na(df: pd.DataFrame, cols: Iterable[str]):
    cols = [c for c in cols if c in df.columns]
    if not cols: return True
    na = df[cols].isna().sum()
    if int(na.sum())>0: raise ValueError(f"NA values present: {{ {', '.join(f'{k}:{int(v)}' for k,v in na.items() if v>0)} }}")
    return True

print("[env] ready")


In [ ]:

ActionReq = Literal["AUTO","TRIAGE_RN","ATTENDING_CONFIRM"]
class PolicyEngine:
    def __init__(self, policy: Dict[str, Any]): self.policy = policy
    def action_requirement(self, triage_level: str, action: str) -> ActionReq:
        return self.policy.get("approvals",{}).get(triage_level,{}).get(action, self.policy.get("global_approvals",{}).get(action,"ATTENDING_CONFIRM"))
    def imaging_protocol(self, key: str) -> Dict[str, Any]: return self.policy.get("imaging_protocols",{}).get(key,{})
    def icu_joker(self) -> Dict[str, Any]: return self.policy.get("icu_joker", {"enabled": False})
    def sla(self) -> Dict[str, Any]: return self.policy.get("sla_targets", {})

policy = yaml.safe_load("""
triage_levels: [red, yellow_green]

global_approvals:
  transfer_to_unit: ATTENDING_CONFIRM

approvals:
  red:
    ecg: AUTO
    labs_standard_ed_panel: AUTO
    ct_resus: AUTO
    ct_non_resus: ATTENDING_CONFIRM
    fast: TRIAGE_RN
    x_ray: TRIAGE_RN
  yellow_green:
    ecg: AUTO
    labs_standard_ed_panel: AUTO
    ct_resus: ATTENDING_CONFIRM
    ct_non_resus: ATTENDING_CONFIRM
    fast: TRIAGE_RN
    x_ray: TRIAGE_RN

orders:
  panels:
    standard_ed_panel: { tests: ["cbc","bmp","lactate","crp","hs-ctnt"] }

imaging_protocols:
  x_ray:
    location: "ED XR room"
    requires_transport: false
    requires_tech_transfer: true
    allowed_on_triage_red: true
    capacity: 1
  ct_resus:
    location: "Resus bay"
    requires_transport: false
    requires_radiology_order: true
    allowed_on_triage_red: true
    capacity: 1
  ct_non_resus:
    location: "Radiology CT"
    requires_transport: true
    requires_radiology_order: true
    allowed_on_triage_red: false
    capacity: 2

icu_joker:
  enabled: true
  trigger_occupancy: 0.9
  extra_beds: 1
  requires_attending: true

sla_targets:
  lab_lactate_minutes: 45
  lab_panel_minutes: 60
  ecg_from_door_minutes: 10
  bed_request_to_decision_minutes: 20
""")
engine = PolicyEngine(policy)
print("[policy] loaded")


In [ ]:

def _coerce_tn(v):
    if v is None: return None
    v = float(v)
    if v < 0 or not (v == v) or v in (float('inf'), float('-inf')): 
        raise ValueError(f"Invalid troponin: {v!r}")
    return v
def hsctnt_99th_url(sex: str="unknown")->float: return 16.8 if sex=="male" else 9.0
def classify_hsctnt_0_1h(t0: float, t1: Optional[float]=None, *, sex: str="unknown", onset_ge_3h: bool=False):
    t0=_coerce_tn(t0); t1=_coerce_tn(t1) if t1 is not None else None
    ri_abs=52.0; ri_d1=5.0; ro_single=5.0; ro_band=12.0; ro_d1=3.0; url=hsctnt_99th_url(sex)
    if t1 is None:
        if t0>=ri_abs:  return "rule_in", {"trigger":"abs_0h","url_99th":url}
        if onset_ge_3h and t0<ro_single: return "rule_out", {"trigger":"single_0h_valid_>3h","url_99th":url}
        return "observe", {"trigger":"needs_1h","url_99th":url}
    delta=t1-t0
    if t0>=ri_abs or delta>=ri_d1: return "rule_in", {"trigger":"abs_0h" if t0>=ri_abs else "delta_1h","url_99th":url}
    if t0<ro_single and onset_ge_3h: return "rule_out", {"trigger":"single_0h_valid_>3h","url_99th":url}
    if t0<ro_band and delta<ro_d1: return "rule_out", {"trigger":"band+delta","url_99th":url}
    return "observe", {"trigger":"observe_zone","url_99th":url}

class GRUMLP(nn.Module):
    def __init__(self, ts_in:int, static_in:int, hidden:int=64, num_layers:int=1, dropout:float=0.1):
        super().__init__()
        self.gru = nn.GRU(ts_in, hidden, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers>1 else 0.0)
        self.static_mlp = nn.Sequential(nn.Linear(static_in, hidden), nn.ReLU(), nn.Dropout(dropout),
                                        nn.Linear(hidden, hidden), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x_ts, x_static, seq_lens=None):
        if seq_lens is not None:
            packed = nn.utils.rnn.pack_padded_sequence(x_ts, seq_lens.cpu(), batch_first=True, enforce_sorted=False)
            _, h = self.gru(packed)
        else:
            _, h = self.gru(x_ts)
        h = h[-1]; s = self.static_mlp(x_static)
        return self.head(torch.cat([h,s], dim=1)).squeeze(1)

def run_inference(x_ts: np.ndarray, x_static: np.ndarray, tn0: float, tn1, sex: str, onset_ge_3h: bool):
    m = GRUMLP(ts_in=x_ts.shape[-1], static_in=x_static.shape[-1], hidden=64).eval()
    with torch.no_grad():
        logits = m(torch.from_numpy(x_ts[None, ...]).float(), torch.from_numpy(x_static[None, ...]).float())
        prob = float(torch.sigmoid(logits).item())
    label, ctx = classify_hsctnt_0_1h(t0=tn0, t1=tn1, sex=sex, onset_ge_3h=onset_ge_3h)
    if label=="rule_in": disp = ("CCU/PCI pathway", f"hs-cTnT {label} via {ctx['trigger']}")
    elif prob>=0.7:     disp = ("ED Observation / Telemetry", f"ML risk {prob:.2f} ≥ 0.70")
    elif label=="rule_out": disp = ("Discharge with early outpatient follow-up", f"hs-cTnT {label} via {ctx['trigger']}")
    else:                disp = ("ED Observation + repeat hs-cTnT at 1h", "Intermediate risk")
    return {"prob": prob, "disposition": disp[0], "reason": disp[1]}
print("[model] ready")


In [ ]:

@dataclass
class EMSAssessment:
    systolic_bp: int; heart_rate: int; spo2: int; gcs: int
    chest_pain: bool=False; suspected_stemi: bool=False; high_trauma: bool=False
    suspected_sepsis: bool=False; pregnancy: bool=False
    def shock_index(self)->float: return self.heart_rate / max(self.systolic_bp,1)
    def triage_level(self)->Literal['red','yellow_green']:
        si = self.shock_index()
        if self.high_trauma or self.suspected_stemi or self.systolic_bp<90 or self.gcs<9 or self.spo2<88 or si>=1.0:
            return 'red'
        return 'yellow_green'
    def fast_track(self)->Optional[str]:
        if self.suspected_stemi: return 'stemi'
        if self.high_trauma: return 'major_trauma'
        return None

@dataclass
class Patient:
    pid: str
    ems: EMSAssessment
    unstable: bool
    lab_results: Dict[str, Any] = field(default_factory=dict)
    board_minutes: int = 0
    destination_unit: Optional[str] = None
    attending_approved: bool = False

@dataclass
class Order:
    kind: Literal['XR','CT','LABS','POCUS','ECG']
    priority: Literal['STAT','ROUTINE']='STAT'
    status: Literal['ordered','awaiting_prereq','awaiting_approval','in_progress','done']='ordered'
    eta_min:int=20; prereq: Optional[List[str]]=None; approval_needed: Optional[str]=None

class DiagnosticsOrchestrator:
    def __init__(self):
        self.queue: List[Order] = []
        self.prereq_done: Dict[str,bool] = {}
    def order(self, kind, priority='STAT', eta_min=20, prereq=None, approval_needed=None):
        status='ordered'
        if approval_needed: status='awaiting_approval'
        elif prereq:
            unmet=[p for p in (prereq or []) if not self.prereq_done.get(p,False)]
            status='awaiting_prereq' if unmet else 'ordered'
        o=Order(kind=kind, priority=priority, eta_min=eta_min, prereq=prereq, approval_needed=approval_needed, status=status)
        self.queue.append(o); return o
    def fulfill_prereq(self, key:str):
        self.prereq_done[key]=True
        for o in self.queue:
            if o.status=='awaiting_prereq' and o.prereq and all(self.prereq_done.get(p,False) for p in o.prereq): o.status='ordered'
    def approve(self, key:str):
        for o in self.queue:
            if o.status=='awaiting_approval' and o.approval_needed==key: o.status='ordered'
    def tick(self, minutes:int=5):
        for o in self.queue:
            if o.status=='ordered': o.status='in_progress'
            elif o.status=='in_progress':
                o.eta_min=max(0,o.eta_min-minutes)
                if o.eta_min==0: o.status='done'
    def status(self)->Dict[str,str]: return {f"{i}:{o.kind}": f"{o.status} ({o.eta_min}m)" for i,o in enumerate(self.queue)}

class LabSystem:
    def __init__(self):
        self.pending: Dict[str, Dict[str,int]] = {}
        self.results: Dict[str, Dict[str,float]] = {}
    def order_panel(self, pid:str, tests: List[str], tat_minutes:int=60):
        self.pending.setdefault(pid, {})
        for t in tests: self.pending[pid][t]=tat_minutes
    def tick(self, minutes:int=5):
        done = []
        for pid, tests in list(self.pending.items()):
            for t, m in list(tests.items()):
                m = max(0, m-minutes); tests[t]=m
                if m==0: done.append((pid,t))
        for pid,t in done:
            self.results.setdefault(pid, {})[t]=float(np.round(np.random.normal(1.0,0.2),3))
            del self.pending[pid][t]
            if not self.pending[pid]: del self.pending[pid]
    def has_results(self, pid:str, test:str)->bool: return pid in self.results and test in self.results[pid]
    def fetch(self, pid:str)->Dict[str,float]: return self.results.get(pid, {})

class ResultsWatcher:
    def __init__(self, lab: LabSystem, sla_minutes:int):
        self.lab=lab; self.sla=sla_minutes; self.time_since_order: Dict[str,int]={}; self.alerts: List[str]=[]
    def start(self, pid:str): self.time_since_order[pid]=0
    def tick(self, pid:str, minutes:int=5):
        self.time_since_order[pid]=self.time_since_order.get(pid,0)+minutes
        if self.time_since_order[pid]>self.sla:
            self.alerts.append(f"SLA breach for {pid}: lab turnaround > {self.sla}m")
    def ready(self, pid:str, needed: List[str])->bool:
        return all(self.lab.has_results(pid, t) for t in needed)

@dataclass
class Hospital:
    name: str; coords: Tuple[float,float]; capabilities: set
    capacity: Dict[str,int] = field(default_factory=lambda: {"ICU":2,"StepDown":2,"EDObs":4,"Ward":20})
    occupied: Dict[str,int] = field(default_factory=dict)
    nurse_slots: Dict[str,int] = field(default_factory=lambda: {"ICU":2,"StepDown":4,"EDObs":4,"Ward":20})
    vents: int = 1; vent_in_use: int = 0
    imaging_protocols: Dict[str,dict] = field(default_factory=dict)
    def available(self, unit: str) -> int: return int(self.capacity.get(unit,0) - self.occupied.get(unit,0))
    def admit(self, unit: str, *, needs_vent: bool=False) -> bool:
        if self.available(unit) <= 0: return False
        if self.nurse_slots.get(unit,0) <= 0: return False
        if needs_vent and (self.vent_in_use >= self.vents): return False
        self.occupied[unit] = self.occupied.get(unit,0) + 1
        self.nurse_slots[unit] = self.nurse_slots.get(unit,0) - 1
        if needs_vent: self.vent_in_use += 1
        return True
    def apply_icu_joker_if_allowed(self, *, attending_ok: bool) -> bool:
        cfg = engine.icu_joker()
        if not cfg.get("enabled"): return False
        occ = self.occupied.get("ICU",0); cap = self.capacity.get("ICU",0)
        ratio = (occ / max(cap,1)) if cap>0 else 1.0
        if ratio >= cfg.get("trigger_occupancy", 0.9) and (attending_ok or not cfg.get("requires_attending", True)):
            self.capacity["ICU"] = self.capacity.get("ICU",0) + int(cfg.get("extra_beds",1))
            self.nurse_slots["ICU"] = self.nurse_slots.get("ICU",0) + int(cfg.get("extra_beds",1))
            return True
        return False

class BedManager:
    def __init__(self, hospital: Hospital, region: List[Hospital]):
        self.h = hospital; self.region = [h for h in region if h.name != hospital.name]
    def disposition_to_level(self, disposition: str) -> Optional[str]:
        if "PCI" in disposition or "CCU" in disposition: return "ICU"
        if "Observation / Telemetry" in disposition: return "StepDown"
        if "Observation + repeat" in disposition: return "EDObs"
        if "Discharge" in disposition: return None
        return "EDObs"
    def request_transfer(self, *, patient: Patient, target_unit: str, attending_ok: bool, needs_vent: bool=False):
        if not attending_ok: return {"approved": False, "reason": "attending approval required"}
        if self.h.admit(target_unit, needs_vent=needs_vent):
            return {"approved": True, "admitted_unit": target_unit, "to_hospital": self.h.name, "transferred": False}
        if target_unit == "ICU" and self.h.apply_icu_joker_if_allowed(attending_ok=True):
            if self.h.admit("ICU", needs_vent=needs_vent):
                return {"approved": True, "admitted_unit": "ICU", "to_hospital": self.h.name, "transferred": False, "joker_used": True}
        for other in self.region:
            if other.admit(target_unit, needs_vent=needs_vent):
                return {"approved": True, "admitted_unit": target_unit, "to_hospital": other.name, "transferred": True}
        return {"approved": True, "admitted_unit": None, "to_hospital": None, "transferred": None, "board_in_ed": True}

class XRTechAgent:
    def __init__(self, hospital: Hospital): self.h = hospital; self.queue: List[str] = []
    def request_transfer(self, pid: str): self.queue.append(pid)
    def tick(self):
        if self.queue: return {"pid": self.queue.pop(0), "status": "xr_done"}
        return None

class TransferCoordinatorAgent:
    def __init__(self, bed_manager: BedManager): self.bm = bed_manager; self.queue: List[Dict[str,Any]] = []
    def request(self, patient: Patient, unit: str, attending_ok: bool, needs_vent: bool=False):
        self.queue.append({"patient": patient, "unit": unit, "attending_ok": attending_ok, "needs_vent": needs_vent})
    def tick(self):
        if not self.queue: return None
        req = self.queue.pop(0)
        res = self.bm.request_transfer(patient=req["patient"], target_unit=req["unit"],
                                       attending_ok=req["attending_ok"], needs_vent=req["needs_vent"])
        return {"pid": req["patient"].pid, **res}

class AttendingAgent:
    def approve_transfer(self, patient: Patient, unit: str)->bool:
        patient.attending_approved = True; return True

def maybe_order(kind, triage_level="yellow_green", *, pregnant=False, life_threatening=False):
    action_map = {"ECG":"ecg","LABS":"labs_standard_ed_panel","POCUS":"fast",
                  "CT":"ct_resus" if triage_level=="red" else "ct_non_resus",
                  "XR":"x_ray"}
    key = action_map.get(kind, kind.lower())
    req = engine.action_requirement(triage_level, key)
    if kind == "CT":
        proto = engine.imaging_protocol("ct_resus" if triage_level=="red" else "ct_non_resus")
        if pregnant and not life_threatening: return "ATTENDING_CONFIRM"
        if proto.get("requires_radiology_order", True): return "requires_ris_order"
        return req
    if kind == "XR":
        proto = engine.imaging_protocol("x_ray")
        if pregnant and not life_threatening: return "ATTENDING_CONFIRM"
        if proto.get("requires_tech_transfer", True): return "requires_tech_transfer"
        return req
    return req

print("[entities] loaded")


In [ ]:

# Region
hA = Hospital("Campus A",(0,0),{"ED","PCI"}, capacity={"ICU":0,"StepDown":1,"EDObs":1,"Ward":10},
              imaging_protocols={"ct_resus":{"allowed_on_triage_red": True},"ct_non_resus":{"allowed_on_triage_red": False},"x_ray":{"requires_tech_transfer": True}})
hB = Hospital("Campus B",(3,1),{"ED","PCI"}, capacity={"ICU":1,"StepDown":1,"EDObs":1,"Ward":12},
              imaging_protocols={"ct_resus":{"allowed_on_triage_red": True},"ct_non_resus":{"allowed_on_triage_red": True},"x_ray":{"requires_tech_transfer": True}})
bm = BedManager(hA, [hA,hB])

# Agents
xr_tech = XRTechAgent(hA)
attending = AttendingAgent()
transfer = TransferCoordinatorAgent(bm)
lab = LabSystem()
watcher = ResultsWatcher(lab, sla_minutes=engine.sla().get("lab_panel_minutes",60))

# Patient: unstable, not fast-track
ems = EMSAssessment(82, 120, 90, 12, chest_pain=False, suspected_sepsis=True)
patient = Patient("ED001", ems, unstable=True)

triage = ems.triage_level()
dx = DiagnosticsOrchestrator()

# Orders
status_ecg = maybe_order("ECG", triage)
status_labs = maybe_order("LABS", triage)
status_xr = maybe_order("XR", "yellow_green", pregnant=False)

if status_xr == "requires_tech_transfer": xr_tech.request_transfer(patient.pid)
panel = policy["orders"]["panels"]["standard_ed_panel"]["tests"]
lab.order_panel(patient.pid, panel, tat_minutes=65)
watcher.start(patient.pid)

timeline = []; approvals=[]

for minute in range(0, 120, 5):
    lab.tick(5); watcher.tick(patient.pid, 5)
    ev = xr_tech.tick()
    if ev: timeline.append({"t":minute, "event":"xr_done", "pid":ev["pid"]})
    if minute==15:
        ok = attending.approve_transfer(patient, unit="ICU"); approvals.append({"t":minute,"unit":"ICU","ok":ok})
        transfer.request(patient, unit="ICU", attending_ok=ok, needs_vent=False)
    move = transfer.tick()
    if move:
        timeline.append({"t":minute, "event":"transfer_attempt", **move})
        if move.get("board_in_ed"): patient.board_minutes += 5
    if watcher.ready(patient.pid, ["cbc","bmp","lactate","crp","hs-ctnt"]):
        if patient.destination_unit is None:
            T=12; x_ts = np.random.rand(T,5).astype("float32"); x_static = np.array([0.70,0,1,0,1], dtype="float32")
            risk = run_inference(x_ts, x_static, tn0=4.0, tn1=None, sex="female", onset_ge_3h=True)
            unit = bm.disposition_to_level(risk["disposition"]) or "EDObs"
            ok = attending.approve_transfer(patient, unit=unit); approvals.append({"t":minute,"unit":unit,"ok":ok})
            transfer.request(patient, unit=unit, attending_ok=ok, needs_vent=False)

for _ in range(10):
    lab.tick(5)
    move = transfer.tick()
    if move: timeline.append({"t":"late", "event":"transfer_attempt", **move})

summary = {
    "triage": triage,
    "orders": {"ECG":status_ecg, "LABS":status_labs, "XR":status_xr},
    "watcher_alerts": watcher.alerts,
    "board_minutes": patient.board_minutes
}
summary


In [ ]:

class RunLogger:
    def __init__(self, base_dir: str="/mnt/data"):
        ts = time.strftime("%Y%m%d-%H%M%S")
        self.run_dir = os.path.join(base_dir, f"ed_run_{ts}")
        os.makedirs(self.run_dir, exist_ok=True)
        self.events: List[Dict[str,Any]] = []
    def _hash(self, obj)->str:
        s = json.dumps(obj, sort_keys=True, default=str).encode()
        return hashlib.sha256(s).hexdigest()[:12]
    def manifest(self, *, policy: Dict[str,Any], seeds: Dict[str,int], env: Dict[str,str]):
        man = {
            "created_at": time.time(),
            "seeds": seeds,
            "env": env,
            "policy_hash": self._hash(policy),
            "policy": policy,
        }
        (Path(self.run_dir)/"manifest.json").write_text(json.dumps(man, indent=2))
    def log_timeline(self, timeline: List[Dict[str,Any]]):
        pd.DataFrame(timeline).to_csv(os.path.join(self.run_dir, "timeline.csv"), index=False)
    def log_approvals(self, approvals: List[Dict[str,Any]]):
        pd.DataFrame(approvals).to_csv(os.path.join(self.run_dir, "approvals.csv"), index=False)
    def log_orders(self, dx: DiagnosticsOrchestrator):
        rec=[asdict(o) for o in dx.queue]
        pd.DataFrame(rec).to_csv(os.path.join(self.run_dir, "orders.csv"), index=False)
    def log_beds(self, h: Hospital, name="home"):
        snap = {"name":name,"capacity":h.capacity,"occupied":h.occupied,"nurse_slots":h.nurse_slots,"vents":h.vents,"vent_in_use":h.vent_in_use}
        (Path(self.run_dir)/f"beds_{name}.json").write_text(json.dumps(snap, indent=2))
    def archive(self)->str:
        zip_path = os.path.join(self.run_dir, "artifacts.zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root,_,files in os.walk(self.run_dir):
                for f in files:
                    if f == "artifacts.zip": continue
                    p = os.path.join(root,f)
                    zf.write(p, arcname=os.path.relpath(p, self.run_dir))
        return zip_path

logger = RunLogger()
logger.manifest(policy=policy, seeds={"python":SEED,"numpy":SEED,"torch":SEED}, env={"torch": torch.__version__})
logger.log_timeline(timeline)
logger.log_approvals(approvals)
logger.log_orders(dx)
logger.log_beds(hA, "home")
logger.log_beds(hB, "region_peer")
zip_path = logger.archive()
print("Artifacts at:", logger.run_dir, "| ZIP:", zip_path)


In [ ]:

print("Notebook saved at:", "/mnt/data/ED_Pathway_Orchestrator_AllInOne_v12.ipynb")
# The artifacts ZIP is inside a timestamped folder; list the parent dir to help users find it.
import os, glob
parents = sorted(glob.glob("/mnt/data/ed_run_*"))
print("Existing run folders:", parents[-3:])
